# 04 — Buffer Visualization Strategies

A buffer is only as useful as the map that shows it. The same data can look clear or confusing depending on three choices:

- **Layer order** — which ring sits on top?
- **Opacity** — how transparent is each fill?
- **Color** — does the palette communicate intensity?

Getting these wrong produces maps where the inner zone is invisible (buried under the outer ring), or where all rings look identical, or where the color implies the opposite of what you mean.

## Setup

In [1]:
import json
import math
from pathlib import Path
from ipyleaflet import Map, GeoJSON, LayersControl, basemaps

DATA_DIR = Path("data")

with open(DATA_DIR / "targets.geojson") as f:
    targets_fc = json.load(f)


def destination_point(lon, lat, bearing_deg, distance_km):
    R = 6371.0
    d = distance_km / R
    phi1 = math.radians(lat)
    lam1 = math.radians(lon)
    theta = math.radians(bearing_deg)
    phi2 = math.asin(math.sin(phi1)*math.cos(d) + math.cos(phi1)*math.sin(d)*math.cos(theta))
    lam2 = lam1 + math.atan2(math.sin(theta)*math.sin(d)*math.cos(phi1),
                              math.cos(d) - math.sin(phi1)*math.sin(phi2))
    return [math.degrees(lam2), math.degrees(phi2)]

def point_buffer(lon, lat, radius_km, n_points=64):
    ring = [destination_point(lon, lat, 360.0*i/n_points, radius_km) for i in range(n_points)]
    ring.append(ring[0])
    return {"type": "Polygon", "coordinates": [ring]}

def buffer_feature(lon, lat, radius_km, name="", n_points=64):
    return {
        "type": "Feature",
        "properties": {"name": name, "radius_km": radius_km},
        "geometry": point_buffer(lon, lat, radius_km, n_points)
    }

def target_coords(name):
    feat = next(f for f in targets_fc["features"] if f["properties"]["name"] == name)
    return feat["geometry"]["coordinates"]

tehran = target_coords("Tehran")
print("Ready.")

Ready.


## Layer Order: Large on Bottom, Small on Top

The most common mistake: adding layers from small to large. The outer ring covers everything added before it, hiding all inner rings.

```
WRONG order  →  outer added last → visible, inner buried under outer
         add inner (50 km)    ← added first, buried
         add middle (150 km)  ← partially buried
         add outer (300 km)   ← added last, covers everything

CORRECT order → outer added first → sits at bottom, inner on top
         add outer (300 km)   ← added first, sits at bottom
         add middle (150 km)  ← on top of outer
         add inner (50 km)    ← on top of both
```

In ipyleaflet, layers are drawn in the order they are added — each new `.add()` call goes on top of previous layers.

In [2]:
zones = [
    {"radius_km": 50,  "label": "Lethal",   "color": "#e74c3c"},
    {"radius_km": 150, "label": "Damaging", "color": "#e67e22"},
    {"radius_km": 300, "label": "Warning",  "color": "#f1c40f"},
]

# --- WRONG: small added first ---
m_wrong = Map(center=(tehran[1], tehran[0]), zoom=4, basemap=basemaps.CartoDB.Positron)
for zone in zones:   # small → large: outer buries inner
    feat = buffer_feature(*tehran, radius_km=zone["radius_km"], name=zone["label"])
    m_wrong.add(GeoJSON(
        data={"type": "FeatureCollection", "features": [feat]},
        name=f"{zone['label']} (WRONG order)",
        style={"color": zone["color"], "fillColor": zone["color"],
               "fillOpacity": 0.55, "weight": 1.5}
    ))

print("Wrong order — inner rings are completely hidden:")
m_wrong

Wrong order — inner rings are completely hidden:


Map(center=[35.695, 51.388], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom…

In [3]:
# --- CORRECT: large added first ---
m_right = Map(center=(tehran[1], tehran[0]), zoom=4, basemap=basemaps.CartoDB.Positron)
for zone in reversed(zones):   # large → small: each inner ring visible on top
    feat = buffer_feature(*tehran, radius_km=zone["radius_km"], name=zone["label"])
    m_right.add(GeoJSON(
        data={"type": "FeatureCollection", "features": [feat]},
        name=f"{zone['label']} (correct order)",
        style={"color": zone["color"], "fillColor": zone["color"],
               "fillOpacity": 0.55, "weight": 1.5}
    ))

print("Correct order — all three rings visible:")
m_right

Correct order — all three rings visible:


Map(center=[35.695, 51.388], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom…

## Opacity Strategy

Even with the right layer order, heavy fills obscure the underlying map. A good opacity strategy:

| Zone | `fillOpacity` | Why |
|---|---|---|
| Outer (low intensity) | 0.10 – 0.20 | Hint of coverage without hiding geography |
| Middle | 0.25 – 0.35 | Visible but still transparent |
| Inner (high intensity) | 0.45 – 0.60 | Solid enough to stand out |

The outer ring should be the *most* transparent — it covers the most area. The inner ring can be more solid because it is small.

Separate `fillOpacity` from border `opacity`. The border (`weight`, `opacity`) can be stronger than the fill — it defines the zone edge without filling the entire area.

In [4]:
# Opacity scaled by zone: outer lightest, inner darkest
opacity_zones = [
    {"radius_km": 50,  "label": "Lethal",   "color": "#e74c3c", "fill_opacity": 0.55},
    {"radius_km": 150, "label": "Damaging", "color": "#e67e22", "fill_opacity": 0.30},
    {"radius_km": 300, "label": "Warning",  "color": "#f1c40f", "fill_opacity": 0.15},
]

m_opacity = Map(center=(tehran[1], tehran[0]), zoom=4, basemap=basemaps.CartoDB.Positron)

for zone in reversed(opacity_zones):
    feat = buffer_feature(*tehran, radius_km=zone["radius_km"], name=zone["label"])
    m_opacity.add(GeoJSON(
        data={"type": "FeatureCollection", "features": [feat]},
        name=f"{zone['label']} ({zone['radius_km']} km)",
        style={
            "color": zone["color"],
            "fillColor": zone["color"],
            "fillOpacity": zone["fill_opacity"],
            "weight": 2,
            "opacity": 0.9
        }
    ))

m_opacity.add(LayersControl())
print("Opacity scaled: outer=0.15, middle=0.30, inner=0.55")
m_opacity

Opacity scaled: outer=0.15, middle=0.30, inner=0.55


Map(center=[35.695, 51.388], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom…

## Color Ramps for Intensity

Color carries meaning. Two common conventions for zone maps:

**Traffic-light** (red = most dangerous, green = safe):
```
inner  →  red     (#e74c3c)  ←  lethal / severe
middle →  orange  (#e67e22)  ←  damaging / moderate
outer  →  yellow  (#f1c40f)  ←  warning / minor
```

**Cool-to-warm** (blue = distant / safe, red = close / dangerous):
```
inner  →  red     (#d62728)
middle →  purple  (#9467bd)
outer  →  blue    (#1f77b4)
```

Avoid using the same hue for all rings — the only differentiator then becomes area, which is hard to judge visually.

In [5]:
# Cool-to-warm palette — compare with traffic light
cool_warm_zones = [
    {"radius_km": 50,  "label": "Core",     "color": "#d62728", "fill_opacity": 0.55},
    {"radius_km": 150, "label": "Mid",      "color": "#9467bd", "fill_opacity": 0.30},
    {"radius_km": 300, "label": "Periphery","color": "#1f77b4", "fill_opacity": 0.15},
]

m_cw = Map(center=(tehran[1], tehran[0]), zoom=4, basemap=basemaps.CartoDB.Positron)

for zone in reversed(cool_warm_zones):
    feat = buffer_feature(*tehran, radius_km=zone["radius_km"], name=zone["label"])
    m_cw.add(GeoJSON(
        data={"type": "FeatureCollection", "features": [feat]},
        name=f"{zone['label']} ({zone['radius_km']} km)",
        style={
            "color": zone["color"],
            "fillColor": zone["color"],
            "fillOpacity": zone["fill_opacity"],
            "weight": 2, "opacity": 0.9
        }
    ))

m_cw.add(LayersControl())
print("Cool-to-warm palette: blue (outer) → purple → red (inner)")
m_cw

Cool-to-warm palette: blue (outer) → purple → red (inner)


Map(center=[35.695, 51.388], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom…

## Border-Only Mode

When the fill is too heavy — especially with overlapping buffers from multiple targets — reduce fill to near-zero and rely on the border alone to communicate zone boundaries.

This is useful for:
- Dense maps with many overlapping zones
- When underlying geography must stay readable
- Reference rings that annotate rather than dominate

In [6]:
riyadh  = target_coords("Riyadh")
madrid  = target_coords("Madrid")
honolulu = target_coords("Honolulu")

all_targets = [
    {"coords": tehran,  "name": "Tehran",   "color": "#e74c3c"},
    {"coords": riyadh,  "name": "Riyadh",   "color": "#e67e22"},
    {"coords": madrid,  "name": "Madrid",   "color": "#2980b9"},
    {"coords": honolulu,"name": "Honolulu", "color": "#27ae60"},
]

RADIUS = 1500   # km — large enough to overlap on a world view

m_border = Map(center=(30, 15), zoom=2, basemap=basemaps.CartoDB.Positron)

for t in all_targets:
    feat = buffer_feature(*t["coords"], radius_km=RADIUS, name=t["name"])
    m_border.add(GeoJSON(
        data={"type": "FeatureCollection", "features": [feat]},
        name=f"{t['name']} {RADIUS} km",
        style={
            "color": t["color"],
            "fillColor": t["color"],
            "fillOpacity": 0.06,   # near-transparent fill
            "weight": 2.5,
            "opacity": 1.0         # solid border
        }
    ))

m_border.add(LayersControl())
print("Border-only style: fill ≈ 0, solid border — readable on a crowded map")
m_border

Border-only style: fill ≈ 0, solid border — readable on a crowded map


Map(center=[30, 15], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_tex…

## Exercise A

Create a **four-zone impact map** around **Riyadh** with radii 30 km, 100 km, 250 km, and 500 km.

1. Assign a distinct color to each zone (your choice of palette — traffic light, cool-to-warm, or custom).
2. Set `fillOpacity` so that the outermost ring is the most transparent and the innermost is the most opaque.
3. Add a `LayersControl` so each zone can be toggled on/off.
4. Print the opacity values you chose and one sentence explaining your palette decision.

In [7]:
from ipyleaflet import Map, GeoJSON, LayersControl, basemaps

riyadh = target_coords("Riyadh")

# 1-2. Four zones with cool-to-warm colors and opacity scaling
zones = [
    {"radius_km": 30,  "label": "Core",      "color": "#d62728", "fillOpacity": 0.60},
    {"radius_km": 100, "label": "High",      "color": "#ff7f0e", "fillOpacity": 0.40},
    {"radius_km": 250, "label": "Moderate",  "color": "#9467bd", "fillOpacity": 0.25},
    {"radius_km": 500, "label": "Outer",     "color": "#1f77b4", "fillOpacity": 0.12},
]

# Create map
m = Map(
    center=(riyadh[1], riyadh[0]),
    zoom=5,
    basemap=basemaps.CartoDB.Positron
)

# Add outermost first so inner zones stay visible
for zone in reversed(zones):
    feat = buffer_feature(
        *riyadh,
        radius_km=zone["radius_km"],
        name=zone["label"]
    )

    m.add(GeoJSON(
        data={
            "type": "FeatureCollection",
            "features": [feat]
        },
        name=f"{zone['label']} ({zone['radius_km']} km)",
        style={
            "color": zone["color"],
            "fillColor": zone["color"],
            "fillOpacity": zone["fillOpacity"],
            "weight": 2,
            "opacity": 0.9
        }
    ))

# 3. Add LayersControl
m.add(LayersControl())

# 4. Print opacity values and palette explanation
print("Opacity values:")
for zone in zones:
    print(f"{zone['label']} ({zone['radius_km']} km): {zone['fillOpacity']}")

print()
print("Palette decision: I used a cool-to-warm palette so the outer zone feels less intense and the inner core stands out as the highest-impact area.")

m

Opacity values:
Core (30 km): 0.6
High (100 km): 0.4
Moderate (250 km): 0.25
Outer (500 km): 0.12

Palette decision: I used a cool-to-warm palette so the outer zone feels less intense and the inner core stands out as the highest-impact area.


Map(center=[24.688, 46.675], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom…

## Exercise B

Display **Tehran** and **Madrid** each with a 1200 km buffer on a world map.

1. Use border-only style (`fillOpacity ≤ 0.08`) with distinct colors per target.
2. Add a 300 km **concentric inner ring** for each target in the same color but with slightly higher opacity.
3. The result should show both targets, both ring sizes, and still allow you to read country names underneath.
4. Describe: do the 1200 km rings overlap? If so, what does that overlap represent geographically?

In [8]:
from ipyleaflet import Map, GeoJSON, LayersControl, basemaps

tehran = target_coords("Tehran")
madrid = target_coords("Madrid")

targets = [
    {"name": "Tehran", "coords": tehran, "color": "#e74c3c"},
    {"name": "Madrid", "coords": madrid, "color": "#2980b9"},
]

m = Map(center=(38, 25), zoom=3, basemap=basemaps.CartoDB.Positron)

for target in targets:
    name = target["name"]
    coords = target["coords"]
    color = target["color"]

    # 1200 km outer ring: border-only style
    outer_feat = buffer_feature(
        *coords,
        radius_km=1200,
        name=f"{name} 1200 km"
    )

    m.add(GeoJSON(
        data={"type": "FeatureCollection", "features": [outer_feat]},
        name=f"{name} 1200 km ring",
        style={
            "color": color,
            "fillColor": color,
            "fillOpacity": 0.06,
            "weight": 2.5,
            "opacity": 1.0
        }
    ))

    # 300 km inner ring: same color, slightly higher opacity
    inner_feat = buffer_feature(
        *coords,
        radius_km=300,
        name=f"{name} 300 km"
    )

    m.add(GeoJSON(
        data={"type": "FeatureCollection", "features": [inner_feat]},
        name=f"{name} 300 km inner ring",
        style={
            "color": color,
            "fillColor": color,
            "fillOpacity": 0.12,
            "weight": 2,
            "opacity": 1.0
        }
    ))

    # Target point
    point_feat = {
        "type": "Feature",
        "properties": {"name": name},
        "geometry": {
            "type": "Point",
            "coordinates": coords
        }
    }

    m.add(GeoJSON(
        data={"type": "FeatureCollection", "features": [point_feat]},
        name=f"{name} point",
        style={
            "color": color,
            "fillColor": color,
            "fillOpacity": 0.9,
            "weight": 2
        }
    ))

m.add(LayersControl())

print("The 1200 km rings do not overlap.")
print("Their centers are too far apart, so there is no shared geographic area within 1200 km of both Tehran and Madrid.")

m

The 1200 km rings do not overlap.
Their centers are too far apart, so there is no shared geographic area within 1200 km of both Tehran and Madrid.


Map(center=[38, 25], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_tex…

---

## Check Your Understanding

**1.** How can poor layer ordering mislead someone reading a buffer map?

**2.** Why is it better to set the *outer* ring to the *lowest* opacity, rather than the other way around?

```python
# No code needed — answer in your own words
```

## Check Your Understanding

### 1.
How can poor layer ordering mislead someone reading a buffer map?

Poor layer ordering can hide important information and make the map appear incorrect even when the data is correct.

For example, if large outer buffers are drawn last, they can completely cover smaller inner zones underneath. A viewer might incorrectly think the inner zones do not exist.

Poor ordering can also:
- hide path lines
- hide overlap regions
- obscure target points
- make boundaries difficult to distinguish

Correct ordering improves readability by keeping the most important or detailed layers visible on top.

---

### 2.
Why is it better to set the *outer* ring to the *lowest* opacity, rather than the other way around?

Outer rings usually cover the largest area, so if they are highly opaque they can dominate the map and hide everything beneath them.

Using low opacity for outer zones:
- keeps country labels visible
- preserves underlying geography
- allows overlap regions to remain readable
- makes inner, higher-priority zones stand out clearly

The innermost zones are usually the most important or highest-impact regions, so they should appear visually stronger with higher opacity.

## Next

In [05 — CRS Limitations](./05-CRS_Limitations.ipynb), we examine why buffers computed in degree offsets are geometrically incorrect — and show exactly how much distortion the naive approach produces at different latitudes.